# Notebook 3: Data Preprocessing

## Objective

Merge all race datasets into one master dataset and prepare it for machine learning.


In [1]:
import os
import glob
import pandas as pd
import numpy as np

In [2]:
# finding all the csv files

csv_files = glob.glob("../data/raw/*.csv")
print(f"Found {len(csv_files)} files")
for file in csv_files:
    print(file)

Found 12 files
../data/raw/2024_Spain_laps.csv
../data/raw/2024_Singapore_laps.csv
../data/raw/2023_Spain_laps.csv
../data/raw/2022_Spain_laps.csv
../data/raw/2024_Bahrain_laps.csv
../data/raw/2022_Singapore_laps.csv
../data/raw/2024_Monza_laps.csv
../data/raw/2023_Singapore_laps.csv
../data/raw/2023_Monza_laps.csv
../data/raw/2023_Bahrain_laps.csv
../data/raw/2022_Monza_laps.csv
../data/raw/2022_Bahrain_laps.csv


In [4]:
# reading every csv file

dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

# merging everything

master_df = pd.concat(
    dataframes,
    ignore_index=True
)
master_df.head()

,Driver,Team,LapNumber,LapTime,Compound,TyreLife,Stint,Position,TrackStatus
0,VER,Red Bull Racing,1.0,0 days 00:01:23.186000,SOFT,4.0,1.0,2.0,1
1,VER,Red Bull Racing,2.0,0 days 00:01:19.871000,SOFT,5.0,1.0,2.0,1
2,VER,Red Bull Racing,3.0,0 days 00:01:19.364000,SOFT,6.0,1.0,1.0,1
3,VER,Red Bull Racing,4.0,0 days 00:01:20.766000,SOFT,7.0,1.0,1.0,1
4,VER,Red Bull Racing,5.0,0 days 00:01:20.827000,SOFT,8.0,1.0,1.0,1


In [5]:
# dataset information
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13308 entries, 0 to 13307
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Driver       13308 non-null  object 
 1   Team         13308 non-null  object 
 2   LapNumber    13308 non-null  float64
 3   LapTime      13151 non-null  object 
 4   Compound     13308 non-null  object 
 5   TyreLife     13308 non-null  float64
 6   Stint        13308 non-null  float64
 7   Position     13286 non-null  float64
 8   TrackStatus  13308 non-null  int64  
dtypes: float64(4), int64(1), object(4)
memory usage: 935.8+ KB


In [6]:
# printing the shape
print(master_df.shape)

(13308, 9)


In [7]:
# finding the missing values

print(master_df.isnull().sum())

Driver           0
Team             0
LapNumber        0
LapTime        157
Compound         0
TyreLife         0
Stint            0
Position        22
TrackStatus      0
dtype: int64


In [8]:
# duplicate row
duplicates = master_df.duplicated().sum()
print("Duplicate rows:", duplicates)

# deleting the duplicates
master_df = master_df.drop_duplicates()
print(master_df.shape)

Duplicate rows: 0
(13308, 9)


In [9]:
# checking the data types

master_df.dtypes

Driver          object
Team            object
LapNumber      float64
LapTime         object
Compound        object
TyreLife       float64
Stint          float64
Position       float64
TrackStatus      int64
dtype: object

In [12]:
# converting the lap time and verifying

master_df["LapTime"] = pd.to_timedelta(master_df["LapTime"])
master_df["LapTimeSeconds"] = master_df["LapTime"].dt.total_seconds()

# verifying
master_df[
    ["LapTime", "LapTimeSeconds"]
].head()

,LapTime,LapTimeSeconds
0,0 days 00:01:23.186000,83.186
1,0 days 00:01:19.871000,79.871
2,0 days 00:01:19.364000,79.364
3,0 days 00:01:20.766000,80.766
4,0 days 00:01:20.827000,80.827


In [13]:
# removing the original laptime
master_df.drop(
    columns=["LapTime"],
    inplace=True
)

In [14]:
# saved the processed dataset
os.makedirs("../data/processed", exist_ok=True)
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)
print("Master dataset saved successfully!")

Master dataset saved successfully!


In [ ]:
# removing the missing values 

master_df = master_df.dropna(
    subset=[
        "LapTimeSeconds",
        "Position"
    ]
)
print(master_df.shape)

#checking
master_df.isnull().sum()

(13151, 9)


Driver            0
Team              0
LapNumber         0
Compound          0
TyreLife          0
Stint             0
Position          0
TrackStatus       0
LapTimeSeconds    0
dtype: int64

In [ ]:
# checking the compounds of tire

master_df["Compound"].value_counts()

Compound
HARD            5058
MEDIUM          4033
SOFT            3534
INTERMEDIATE     526
Name: count, dtype: int64

In [20]:
# checking the track status
master_df["TrackStatus"].value_counts().sort_index()


TrackStatus
1       12232
2          15
4         169
6           9
12        391
14         17
16          4
21         41
24         30
26         16
41         50
64          4
67          1
71          9
124         9
126        48
164        15
216         3
671        53
712         2
1267        4
2167        1
2671       28
Name: count, dtype: int64

In [21]:
# basic stats
master_df.describe()

,LapNumber,TyreLife,Stint,Position,TrackStatus,LapTimeSeconds
count,13151.000000,13151.000000,13151.000000,13151.000000,13151.000000,13151.000000
mean,29.612349,12.745647,2.026766,9.967987,11.579119,94.375429
std,17.371573,8.262114,0.889507,5.523496,133.881938,12.334353
min,1.000000,1.000000,1.000000,1.000000,1.000000,76.330000
25%,15.000000,6.000000,1.000000,5.000000,1.000000,85.607000
50%,29.000000,11.000000,2.000000,10.000000,1.000000,95.376000
75%,44.000000,18.000000,3.000000,15.000000,1.000000,99.577500
max,66.000000,50.000000,7.000000,20.000000,2671.000000,149.966000


In [19]:
# saving again
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

In [22]:
# now we will only keep the dry compounds

master_df = master_df[
    master_df["Compound"].isin([
        "SOFT",
        "MEDIUM",
        "HARD"
    ])
]
print(master_df.shape)

(12625, 9)


In [23]:
# keeping only the green flag laps
master_df = master_df[
    master_df["TrackStatus"] == 1
]
print(master_df.shape)

(11880, 9)


In [ ]:
master_df["Compound"].value_counts()

Compound
HARD      4896
MEDIUM    3845
SOFT      3139
Name: count, dtype: int64

In [ ]:
master_df["TrackStatus"].value_counts()

TrackStatus
1    11880
Name: count, dtype: int64

In [26]:
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)

print(master_df.shape)

(11880, 9)
